### 1. Mount Google Drive and Extract Dataset
First, we mount Google Drive to access the dataset (`skindata.zip` or similar) and extract it to the local Colab runtime environment.

In [ ]:
from google.colab import drive
import zipfile
import os

# Mount Google Drive
drive.mount('/content/drive')

# Base paths pointing to the exact '01.원천데이터' directories inside Training and Validation
training_base_path = '/content/drive/MyDrive/skindata/3.개방데이터/1.데이터/Validation/Training/01.원천데이터'
validation_base_path = '/content/drive/MyDrive/skindata/3.개방데이터/1.데이터/Validation/01.원천데이터'

extract_path = '/content/skindata'
os.makedirs(extract_path, exist_ok=True)

def extract_zips_in_folder(source_dir, target_subfolder):
    target_dir = os.path.join(extract_path, target_subfolder)
    os.makedirs(target_dir, exist_ok=True)

    if os.path.exists(source_dir):
        files = os.listdir(source_dir)
        zip_files = [f for f in files if f.endswith('.zip')]

        if not zip_files:
            print(f"{source_dir} 폴더 내에 zip 파일이 존재하지 않습니다.")
            return

        for zip_file in zip_files:
            full_zip_path = os.path.join(source_dir, zip_file)
            print(f"Extracting {zip_file} to {target_dir}...")
            with zipfile.ZipFile(full_zip_path, 'r') as zip_ref:
                zip_ref.extractall(target_dir)
        print(f"{source_dir} extraction completed successfully.\n")
    else:
        print(f"경로를 찾을 수 없습니다: {source_dir}")

# Extract zip files from '01.원천데이터' folders
extract_zips_in_folder(training_base_path, 'train')
extract_zips_in_folder(validation_base_path, 'val')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting TS_광선각화증.zip to /content/skindata/train...
Extracting TS_기저세포암.zip to /content/skindata/train...
Extracting TS_멜라닌세포모반.zip to /content/skindata/train...
Extracting TS_보웬병.zip to /content/skindata/train...
Extracting TS_비립종.zip to /content/skindata/train...
Extracting TS_사마귀.zip to /content/skindata/train...
Extracting TS_악성흑색종.zip to /content/skindata/train...
Extracting TS_지루각화증.zip to /content/skindata/train...
Extracting TS_편평세포암.zip to /content/skindata/train...
Extracting TS_표피낭종.zip to /content/skindata/train...
Extracting TS_피부섬유종.zip to /content/skindata/train...
Extracting TS_피지샘증식증.zip to /content/skindata/train...
Extracting TS_혈관종.zip to /content/skindata/train...
Extracting TS_화농 육아종.zip to /content/skindata/train...
Extracting TS_흑색

### 2. Prepare DataLoaders with Transforms and 80:20 Train/Val Split
We will define the ImageNet transformations (resizing to 300x300, normalizations) and split the training folder into Train and Validation sets (8:2 ratio).

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import os

# GPU configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Image transformations (Resize/Crop to 300x300, Normalized for ImageNet)
train_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Set paths to train and val
train_dir = os.path.join(extract_path, 'train')
val_dir = os.path.join(extract_path, 'val')

# Advanced Robust Helper to locate the exact parent folder of class folders containing images
def find_actual_root(base_dir):
    for root, dirs, files in os.walk(base_dir):
        # Exclude hidden directories/system files
        dirs = [d for d in dirs if not d.startswith('.')]
        # Find the directory that contains multiple child directories representing the classes
        if len(dirs) >= 2:
            # Verify if at least one of these subdirs contains image files
            for d in dirs:
                sub_path = os.path.join(root, d)
                if os.path.isdir(sub_path):
                    sub_files = [f for f in os.listdir(sub_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                    if len(sub_files) > 0:
                        return root
    return base_dir

# Auto-detect real class roots to handle nested zip structures dynamically
resolved_train_dir = find_actual_root(train_dir)
resolved_val_dir = find_actual_root(val_dir)

print(f"Resolved Training Class Root: {resolved_train_dir}")
print(f"Resolved Validation Class Root: {resolved_val_dir}")

# Load datasets using the resolved class directories
full_train_dataset = datasets.ImageFolder(root=resolved_train_dir, transform=train_transforms)

# 80:20 Train / Test split from the Training dataset as requested
train_size = int(0.8 * len(full_train_dataset))
test_size = len(full_train_dataset) - train_size

train_dataset, test_dataset = random_split(full_train_dataset, [train_size, test_size])
# Apply test transforms to the 20% split test dataset
test_dataset.dataset.transform = val_test_transforms

# Use 'Validation' extracted folder as our validation dataset
val_dataset = datasets.ImageFolder(root=resolved_val_dir, transform=val_test_transforms)

# DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"\nTrain samples (80%): {len(train_dataset)}")
print(f"Test samples (20%): {len(test_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Using device: cuda
Resolved Training Class Root: /content/skindata/train
Resolved Validation Class Root: /content/skindata/val

Train samples (80%): 9600
Test samples (20%): 2400
Validation samples: 1500


### 3. Model Architecture (EfficientNet-B3 with Custom Classifier Head)
We load pretrained `efficientnet_b3`, replace its classification head with the custom configuration (Global Average Pooling -> Dropout -> Linear with 15 outputs).

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

def create_model(num_classes=15, dropout_rate=0.3):
    # Load pretrained EfficientNet-B3
    weights = EfficientNet_B3_Weights.DEFAULT
    model = efficientnet_b3(weights=weights)

    # EfficientNet-B3 features end with AdaptiveAvgPool2d (Global Average Pooling)
    # We modify the classifier (Dropout -> Linear)
    in_features = model.classifier[1].in_features

    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate, inplace=True),
        nn.Linear(in_features, num_classes)
        # Note: Softmax is implicitly applied within CrossEntropyLoss during training
    )
    return model

model = create_model(num_classes=15).to(device)
print(model.classifier)

Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1536, out_features=15, bias=True)
)


### 4. Training and Evaluation Routine
We define the loss function, optimizer, and training loop, followed by validation and final test accuracy evaluation.

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total

# Train for 5 epochs as a starter
epochs = 5
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")

# Final Evaluation on Test Set
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"\n[Final Results] Test Accuracy: {test_acc*100:.2f}%")

Epoch 1/5 - Train Loss: 0.6165, Train Acc: 89.47% | Val Loss: 0.0187, Val Acc: 99.73%
Epoch 2/5 - Train Loss: 0.0358, Train Acc: 99.15% | Val Loss: 0.0090, Val Acc: 99.73%
